# Age Detection Pipeline: Under-18 vs Adult Classification

## Complete Pipeline with Maximum Image Size (584×584) and DDP

**Configuration:**
- **Image Size:** 584×584 (Maximum for EfficientNet-B3)
- **Model:** EfficientNet-B3 (Pre-trained on ImageNet)
- **Training:** Distributed Data Parallel (DDP) for multi-GPU
- **Dataset:** UTKFace (23,708 images)
- **Task:** Binary Classification (Under-18 vs Adult)
- **Primary Metric:** Under-18 Recall (minimize false negatives)

---


## Step 1: Environment Setup & Library Imports


In [ ]:
# ============================================================
# STEP 1: ENVIRONMENT SETUP & LIBRARY IMPORTS
# ============================================================

# Core Python libraries
import os
import sys
import time
import copy
from collections import defaultdict

# Data manipulation
import pandas as pd
import numpy as np

# Image processing
from PIL import Image
import cv2

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch core
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Distributed training
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

# Torchvision
import torchvision
from torchvision import transforms, models
import torchvision.utils as vutils

# Sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)

# Progress bars
from tqdm.notebook import tqdm

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

# Better plot quality
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*60)
print("✓ All libraries imported successfully")
print("="*60)


## Step 2: GPU Configuration & DDP Setup


In [ ]:
# ============================================================
# STEP 2: GPU CONFIGURATION & DDP SETUP
# ============================================================

print("="*60)
print("GPU CONFIGURATION")
print("="*60)

if not torch.cuda.is_available():
    print("❌ CUDA not available! Check GPU settings.")
    sys.exit()

print(f"✓ CUDA Available: {torch.cuda.is_available()}")
print(f"✓ PyTorch Version: {torch.__version__}")
print(f"✓ CUDA Version: {torch.version.cuda}")
print(f"✓ Number of GPUs: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Total Memory: {props.total_memory / 1e9:.2f} GB")
    print(f"  CUDA Capability: {props.major}.{props.minor}")
    print(f"  Multi-Processors: {props.multi_processor_count}")

# DDP Setup
def setup_ddp(force_ddp=False):
    """
    Initialize distributed training
    
    For Kaggle notebooks:
    - If DDP env vars exist, use proper DDP
    - If force_ddp=True and multiple GPUs, manually initialize DDP (notebook mode)
    - If multiple GPUs detected but no env vars, use DataParallel (notebook-friendly)
    - Otherwise use single GPU
    
    Args:
        force_ddp: If True, manually initialize DDP even without env vars (notebook mode)
    """
    num_gpus = torch.cuda.device_count()
    
    # Check if DDP environment variables are set (proper DDP mode)
    if 'RANK' in os.environ and 'WORLD_SIZE' in os.environ and 'LOCAL_RANK' in os.environ:
        rank = int(os.environ['RANK'])
        world_size = int(os.environ['WORLD_SIZE'])
        local_rank = int(os.environ['LOCAL_RANK'])
        use_ddp = True
        
        if world_size > 1:
            dist.init_process_group(
                backend='nccl',
                init_method='env://'
            )
            torch.cuda.set_device(local_rank)
            device = torch.device(f'cuda:{local_rank}')
            print(f"\n✓ DDP Initialized: Rank {rank}/{world_size-1}, Local Rank {local_rank}")
        else:
            device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
            use_ddp = False
            print(f"\n✓ Single GPU Mode: {device}")
    elif force_ddp and num_gpus > 1:
        # Manually initialize DDP for notebook (not recommended but possible)
        # This is a workaround for notebooks - proper DDP requires process spawning
        print(f"\n⚠ Attempting manual DDP initialization (notebook mode)")
        print(f"  Note: True DDP requires separate processes. This is a workaround.")
        
        # Set environment variables manually
        os.environ['MASTER_ADDR'] = 'localhost'
        os.environ['MASTER_PORT'] = '12355'
        os.environ['RANK'] = '0'
        os.environ['WORLD_SIZE'] = str(num_gpus)
        os.environ['LOCAL_RANK'] = '0'
        
        try:
            # Initialize process group
            dist.init_process_group(
                backend='nccl',
                init_method='env://',
                world_size=num_gpus,
                rank=0
            )
            torch.cuda.set_device(0)
            device = torch.device("cuda:0")
            rank = 0
            world_size = num_gpus
            local_rank = 0
            use_ddp = True
            print(f"✓ Manual DDP initialized (limited - only rank 0 active)")
            print(f"  ⚠ Warning: This is NOT true DDP. Use DataParallel for better notebook compatibility.")
        except Exception as e:
            print(f"  ❌ Manual DDP failed: {e}")
            print(f"  Falling back to DataParallel...")
            rank = 0
            world_size = num_gpus
            local_rank = 0
            use_ddp = False
            device = torch.device("cuda:0")
    else:
        # No DDP env vars - check if we have multiple GPUs
        if num_gpus > 1:
            # Use DataParallel for notebook compatibility
            rank = 0
            world_size = num_gpus  # For LR scaling purposes
            local_rank = 0
            use_ddp = False  # Will use DataParallel instead
            device = torch.device("cuda:0")
            print(f"\n✓ Multiple GPUs detected ({num_gpus}) but DDP env vars not set")
            print(f"  Will use DataParallel for multi-GPU training (notebook-friendly)")
            print(f"  💡 To use DDP: Set force_ddp=True or use torchrun/script execution")
        else:
            # Single GPU
            rank = 0
            world_size = 1
            local_rank = 0
            use_ddp = False
            device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
            print(f"\n✓ Single GPU Mode: {device}")
    
    return device, rank, world_size, local_rank, use_ddp

# Set to True to attempt manual DDP (not recommended - DataParallel is better for notebooks)
FORCE_DDP = False  # Change to True if you want to try manual DDP initialization

device, rank, world_size, local_rank, use_ddp = setup_ddp(force_ddp=FORCE_DDP)

device, rank, world_size, local_rank, use_ddp = setup_ddp()

# Set primary device for non-distributed operations
if world_size == 1:
    primary_device = device
else:
    primary_device = torch.device(f'cuda:{local_rank}')

print(f"✓ Primary device: {primary_device}")
print(f"✓ Training will use: {'DDP' if use_ddp else ('DataParallel' if world_size > 1 else 'Single GPU')}")
print("="*60)


## Step 3: Project Configuration


In [ ]:
# ============================================================
# STEP 3: PROJECT CONFIGURATION
# ============================================================

class Config:
    """Centralized configuration"""
    
    # Paths - UTKFace dataset has two directories
    DATASET_PATH_1 = '/kaggle/input/utkface-new/UTKFace'
    DATASET_PATH_2 = '/kaggle/input/utkface-new/crop_part1'
    OUTPUT_DIR = '/kaggle/working/'
    
    # Random seed
    RANDOM_SEED = 42
    
    # Image settings - MAXIMUM SIZE
    IMAGE_SIZE = 584  # Maximum practical size for EfficientNet-B3
    
    # Model settings
    MODEL_NAME = 'efficientnet_b3'
    NUM_CLASSES = 2  # Under-18 vs Adult
    PRETRAINED = True
    
    # Training settings
    BATCH_SIZE = 32  # Per GPU (increased for better GPU utilization with 584×584)
    NUM_EPOCHS = 60
    BASE_LEARNING_RATE = 0.001  # Base LR for single GPU
    WEIGHT_DECAY = 0.0001
    
    def __init__(self, num_gpus=1):
        """
        Initialize config with GPU-aware learning rate scaling
        
        Args:
            num_gpus: Number of GPUs (for LR scaling)
            
        Note:
            Learning rate is scaled linearly with number of GPUs.
            This follows the linear scaling rule: LR = base_lr × num_gpus
            This maintains the same effective learning rate per sample when
            using distributed training with larger effective batch sizes.
            
            Example:
                - 1 GPU:  LR = 0.001 × 1 = 0.001
                - 2 GPUs: LR = 0.001 × 2 = 0.002
                - 4 GPUs: LR = 0.001 × 4 = 0.004
        """
        # Scale learning rate linearly with number of GPUs
        # Linear scaling rule: LR = base_lr * num_gpus
        # This maintains the same effective learning rate per sample
        self.LEARNING_RATE = self.BASE_LEARNING_RATE * num_gpus
    
    # Data split
    TRAIN_RATIO = 0.70
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15
    
    # DataLoader
    NUM_WORKERS = 4
    PIN_MEMORY = True
    
    # Optimizer & Scheduler
    OPTIMIZER = 'adamw'
    SCHEDULER = 'reduce_on_plateau'
    SCHEDULER_PATIENCE = 5
    SCHEDULER_FACTOR = 0.5
    SCHEDULER_MIN_LR = 1e-6
    
    # Early stopping
    EARLY_STOPPING_PATIENCE = 12
    
    # Mixed precision
    USE_AMP = True  # Automatic Mixed Precision (FP16)
    
    # Checkpointing
    SAVE_BEST_ONLY = True
    METRIC_TO_MONITOR = 'val_under18_recall'  # Primary metric
    
    # Class names
    CLASS_NAMES = ['under18', 'adult']
    
    # Age threshold
    AGE_THRESHOLD = 18

# Initialize config with GPU count for LR scaling
config = Config(num_gpus=world_size)

print("="*60)
print("PROJECT CONFIGURATION")
print("="*60)
print(f"Image Size: {config.IMAGE_SIZE}×{config.IMAGE_SIZE} (MAXIMUM)")
print(f"Model: {config.MODEL_NAME}")
print(f"Batch Size: {config.BATCH_SIZE} (per GPU)")
print(f"Total Batch Size: {config.BATCH_SIZE * world_size} (with {world_size} GPUs)")
print(f"Epochs: {config.NUM_EPOCHS}")
print(f"Base Learning Rate: {config.BASE_LEARNING_RATE}")
print(f"Scaled Learning Rate: {config.LEARNING_RATE} (×{world_size} for {world_size} GPU{'s' if world_size > 1 else ''})")
print(f"LR Scaling: Linear (LR = base_lr × num_gpus)")
print(f"Mixed Precision: {config.USE_AMP}")
training_mode = 'DDP' if (world_size > 1 and use_ddp) else ('DataParallel' if world_size > 1 else 'Single GPU')
print(f"Training Mode: {training_mode}")
print(f"Primary Metric: {config.METRIC_TO_MONITOR}")
print("="*60)


## Step 4: Dataset Loading & Verification


In [ ]:
# ============================================================
# STEP 4: DATASET LOADING & VERIFICATION
# ============================================================

print("="*60)
print("DATASET VERIFICATION")
print("="*60)

# Check both dataset paths
dataset_paths = []
if os.path.exists(config.DATASET_PATH_1):
    dataset_paths.append(config.DATASET_PATH_1)
    print(f"✓ Found: {config.DATASET_PATH_1}")
else:
    print(f"⚠ Not found: {config.DATASET_PATH_1}")

if os.path.exists(config.DATASET_PATH_2):
    dataset_paths.append(config.DATASET_PATH_2)
    print(f"✓ Found: {config.DATASET_PATH_2}")
else:
    print(f"⚠ Not found: {config.DATASET_PATH_2}")

if len(dataset_paths) == 0:
    print("❌ No dataset paths found!")
    print("Available paths:")
    if os.path.exists('/kaggle/input/'):
        for path in os.listdir('/kaggle/input/'):
            print(f"  - /kaggle/input/{path}")
    sys.exit()

# Collect all image files from both directories
image_files = []
for dataset_path in dataset_paths:
    files = [f for f in os.listdir(dataset_path) 
             if f.endswith(('.jpg', '.jpeg', '.png'))]
    image_files.extend([(f, dataset_path) for f in files])
    print(f"  Found {len(files)} images in {dataset_path}")

print(f"\n✓ Total images: {len(image_files):,}")
print(f"✓ Sample files:")
for i in range(min(5, len(image_files))):
    filename, path = image_files[i]
    print(f"    {filename} (from {os.path.basename(path)})")
print("="*60)


## Step 5: Data Parsing & Labeling


In [ ]:
# ============================================================
# STEP 5: DATA PARSING & LABELING
# ============================================================

def parse_utkface_filename(filename):
    """
    Parse UTKFace filename format: [age]_[gender]_[race]_[date].jpg
    
    Args:
        filename: str, filename to parse
        
    Returns:
        dict with age, gender, race, or None if parsing fails
    """
    try:
        parts = filename.split('_')
        age = int(parts[0])
        gender = int(parts[1])  # 0=male, 1=female
        race = int(parts[2])     # 0=white, 1=black, 2=asian, 3=indian, 4=others
        
        return {
            'age': age,
            'gender': gender,
            'race': race
        }
    except:
        return None

# Test parsing
if len(image_files) > 0:
    test_filename, _ = image_files[0]
    parsed = parse_utkface_filename(test_filename)
    print(f"Test filename: {test_filename}")
    print(f"Parsed info: {parsed}")

print("\nParsing all filenames...")

data_list = []

for filename, dataset_path in tqdm(image_files, desc="Processing files"):
    parsed = parse_utkface_filename(filename)
    
    if parsed is not None:
        age = parsed['age']
        
        # Create binary label
        label = 0 if age < config.AGE_THRESHOLD else 1
        label_name = config.CLASS_NAMES[label]
        
        # Full file path
        filepath = os.path.join(dataset_path, filename)
        
        # Append to list
        data_list.append({
            'filename': filename,
            'filepath': filepath,
            'age': age,
            'gender': parsed['gender'],
            'race': parsed['race'],
            'label': label,
            'label_name': label_name
        })

# Create DataFrame
df = pd.DataFrame(data_list)

print(f"\n✓ Successfully parsed: {len(df):,} images")
print(f"✓ Failed to parse: {len(image_files) - len(df)} images")


In [ ]:
# ============================================================
# STEP 6: DATASET STATISTICS & VISUALIZATION
# ============================================================

print("="*60)
print("DATASET STATISTICS")
print("="*60)

# Basic info
print(f"\nTotal samples: {len(df):,}")
print(f"\nAge statistics:")
print(df['age'].describe())

# Class distribution
print(f"\nClass distribution:")
class_counts = df['label_name'].value_counts()
print(class_counts)
if len(class_counts) == 2:
    print(f"\nImbalance ratio: {class_counts['adult'] / class_counts['under18']:.2f}x")

# Age group breakdown
print(f"\nAge group breakdown:")
age_groups = {
    '0-5 (Babies/Toddlers)': (0, 5),
    '6-10 (Young Children)': (6, 10),
    '11-14 (Pre-teens)': (11, 14),
    '15-17 (Adolescents)': (15, 17),
    '18-25 (Young Adults)': (18, 25),
    '26-40 (Adults)': (26, 40),
    '41-60 (Middle Age)': (41, 60),
    '61+ (Seniors)': (61, 200)
}

for group_name, (min_age, max_age) in age_groups.items():
    count = len(df[(df['age'] >= min_age) & (df['age'] <= max_age)])
    pct = count / len(df) * 100
    print(f"  {group_name:25s}: {count:5d} ({pct:5.2f}%)")

# Gender distribution
print(f"\nGender distribution:")
print(df['gender'].value_counts())

# Race distribution
print(f"\nRace distribution:")
print(df['race'].value_counts())

print("="*60)


In [ ]:
# Visualization
fig = plt.figure(figsize=(18, 12))

# 1. Age histogram with threshold
ax1 = plt.subplot(3, 3, 1)
plt.hist(df['age'], bins=60, edgecolor='black', alpha=0.7)
plt.axvline(x=config.AGE_THRESHOLD, color='red', linestyle='--', 
            linewidth=2, label=f'Age {config.AGE_THRESHOLD} threshold')
plt.xlabel('Age', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Age Distribution', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)

# 2. Class distribution
ax2 = plt.subplot(3, 3, 2)
class_counts.plot(kind='bar', color=['#FF6B6B', '#4ECDC4'])
plt.xlabel('Class', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Class Distribution', fontsize=14, fontweight='bold')
plt.xticks(rotation=0)
plt.grid(alpha=0.3)

# 3. Age groups
ax3 = plt.subplot(3, 3, 3)
age_group_data = []
for group_name, (min_age, max_age) in age_groups.items():
    count = len(df[(df['age'] >= min_age) & (df['age'] <= max_age)])
    age_group_data.append((group_name.split('(')[0].strip(), count))

groups, counts = zip(*age_group_data)
plt.barh(groups, counts, color='skyblue', edgecolor='black')
plt.xlabel('Count', fontsize=12)
plt.title('Age Group Distribution', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3, axis='x')

# 4. Under-18 age distribution
ax4 = plt.subplot(3, 3, 4)
under18_ages = df[df['label'] == 0]['age']
plt.hist(under18_ages, bins=18, edgecolor='black', alpha=0.7, color='#FF6B6B')
plt.xlabel('Age', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Under-18 Age Distribution', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)

# 5. Adult age distribution
ax5 = plt.subplot(3, 3, 5)
adult_ages = df[df['label'] == 1]['age']
plt.hist(adult_ages, bins=50, edgecolor='black', alpha=0.7, color='#4ECDC4')
plt.xlabel('Age', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Adult Age Distribution', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)

# 6. Gender by class
ax6 = plt.subplot(3, 3, 6)
gender_class = pd.crosstab(df['label_name'], df['gender'])
gender_class.columns = ['Male', 'Female']
gender_class.plot(kind='bar', ax=ax6, color=['#95E1D3', '#F38181'])
plt.xlabel('Class', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Gender Distribution by Class', fontsize=14, fontweight='bold')
plt.xticks(rotation=0)
plt.legend(title='Gender')
plt.grid(alpha=0.3)

# 7. Race distribution
ax7 = plt.subplot(3, 3, 7)
race_names = {0: 'White', 1: 'Black', 2: 'Asian', 3: 'Indian', 4: 'Others'}
race_counts = df['race'].value_counts().sort_index()
race_labels = [race_names[i] for i in race_counts.index]
plt.pie(race_counts.values, labels=race_labels, autopct='%1.1f%%', startangle=90)
plt.title('Race Distribution', fontsize=14, fontweight='bold')

# 8. Age vs Class boxplot
ax8 = plt.subplot(3, 3, 8)
df.boxplot(column='age', by='label_name', ax=ax8)
plt.xlabel('Class', fontsize=12)
plt.ylabel('Age', fontsize=12)
plt.title('Age Distribution by Class', fontsize=14, fontweight='bold')
plt.suptitle('')  # Remove default title
plt.grid(alpha=0.3)

# 9. Cumulative age distribution
ax9 = plt.subplot(3, 3, 9)
sorted_ages = np.sort(df['age'])
cumulative = np.arange(1, len(sorted_ages) + 1) / len(sorted_ages) * 100
plt.plot(sorted_ages, cumulative, linewidth=2)
plt.axvline(x=config.AGE_THRESHOLD, color='red', linestyle='--', 
            linewidth=2, label=f'Age {config.AGE_THRESHOLD}')
plt.xlabel('Age', fontsize=12)
plt.ylabel('Cumulative %', fontsize=12)
plt.title('Cumulative Age Distribution', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR, 'dataset_analysis.png'), 
            dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved: dataset_analysis.png")


## Step 7: Dataset Balancing


In [ ]:
# ============================================================
# STEP 7: DATASET BALANCING
# ============================================================

print("="*60)
print("DATASET BALANCING")
print("="*60)

# Separate by class
under18_df = df[df['label'] == 0].copy()
adult_df = df[df['label'] == 1].copy()

print(f"\nBefore balancing:")
print(f"  Under-18: {len(under18_df):,} samples")
print(f"  Adult: {len(adult_df):,} samples")
if len(under18_df) > 0 and len(adult_df) > 0:
    print(f"  Ratio: {len(adult_df) / len(under18_df):.2f}x imbalance")

# Determine target size (minority class size)
target_size = min(len(under18_df), len(adult_df))

print(f"\nBalancing to {target_size:,} samples per class...")

# Undersample majority class
if len(adult_df) > len(under18_df):
    adult_balanced = resample(
        adult_df,
        n_samples=target_size,
        random_state=config.RANDOM_SEED,
        replace=False
    )
    under18_balanced = under18_df.copy()
else:
    under18_balanced = resample(
        under18_df,
        n_samples=target_size,
        random_state=config.RANDOM_SEED,
        replace=False
    )
    adult_balanced = adult_df.copy()

# Combine
balanced_df = pd.concat([under18_balanced, adult_balanced], ignore_index=True)

# Shuffle
balanced_df = balanced_df.sample(frac=1, random_state=config.RANDOM_SEED).reset_index(drop=True)

print(f"\nAfter balancing:")
print(f"  Total samples: {len(balanced_df):,}")
print(balanced_df['label_name'].value_counts())

# Verify perfect balance
is_balanced = len(balanced_df[balanced_df['label']==0]) == len(balanced_df[balanced_df['label']==1])
print(f"  Perfectly balanced: {is_balanced} ✓" if is_balanced else f"  Balanced: {is_balanced} ✗")

# Save balanced dataset
balanced_df.to_csv(os.path.join(config.OUTPUT_DIR, 'balanced_dataset.csv'), index=False)
print(f"\n✓ Balanced dataset saved")
print("="*60)


## Step 8: Train-Val-Test Split


In [ ]:
# ============================================================
# STEP 8: TRAIN-VAL-TEST SPLIT
# ============================================================

print("="*60)
print("TRAIN-VAL-TEST SPLIT")
print("="*60)

# First split: train vs temp (70% vs 30%)
train_df, temp_df = train_test_split(
    balanced_df,
    test_size=(config.VAL_RATIO + config.TEST_RATIO),
    stratify=balanced_df['label'],
    random_state=config.RANDOM_SEED
)

# Second split: val vs test (15% vs 15%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=config.TEST_RATIO / (config.VAL_RATIO + config.TEST_RATIO),
    stratify=temp_df['label'],
    random_state=config.RANDOM_SEED
)

print(f"\nSplit completed:")
print(f"\nTraining set: {len(train_df):,} images ({config.TRAIN_RATIO*100:.0f}%)")
print(train_df['label_name'].value_counts())

print(f"\nValidation set: {len(val_df):,} images ({config.VAL_RATIO*100:.0f}%)")
print(val_df['label_name'].value_counts())

print(f"\nTest set: {len(test_df):,} images ({config.TEST_RATIO*100:.0f}%)")
print(test_df['label_name'].value_counts())

# Verify splits
total = len(balanced_df)
print(f"\nVerification:")
print(f"  Train %: {len(train_df)/total*100:.2f}%")
print(f"  Val %: {len(val_df)/total*100:.2f}%")
print(f"  Test %: {len(test_df)/total*100:.2f}%")
print(f"  Total check: {len(train_df) + len(val_df) + len(test_df) == total} ✓")

# Save splits
train_df.to_csv(os.path.join(config.OUTPUT_DIR, 'train_split.csv'), index=False)
val_df.to_csv(os.path.join(config.OUTPUT_DIR, 'val_split.csv'), index=False)
test_df.to_csv(os.path.join(config.OUTPUT_DIR, 'test_split.csv'), index=False)

print(f"\n✓ Split CSVs saved")
print("="*60)


## Step 9: Data Transforms (584×584)


In [ ]:
# ============================================================
# STEP 9: DATA TRANSFORMS (584×584 MAXIMUM SIZE)
# ============================================================

print("="*60)
print("DATA TRANSFORMS")
print("="*60)

# Training transforms with heavy augmentation for 584×584
train_transforms = transforms.Compose([
    # Resize to slightly larger for cropping
    transforms.Resize((640, 640)),
    
    # Random resized crop
    transforms.RandomResizedCrop(
        config.IMAGE_SIZE,  # 584×584
        scale=(0.85, 1.0),  # Crop 85-100%
        ratio=(0.95, 1.05)  # Near-square
    ),
    
    # Random horizontal flip
    transforms.RandomHorizontalFlip(p=0.5),
    
    # Random rotation
    transforms.RandomRotation(
        degrees=12,
        fill=0
    ),
    
    # Color augmentation
    transforms.ColorJitter(
        brightness=0.25,
        contrast=0.25,
        saturation=0.20,
        hue=0.05
    ),
    
    # Random Gaussian blur
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
    ], p=0.25),
    
    # Random grayscale
    transforms.RandomGrayscale(p=0.1),
    
    # Random perspective
    transforms.RandomPerspective(distortion_scale=0.2, p=0.2),
    
    # Convert to tensor
    transforms.ToTensor(),
    
    # Normalize with ImageNet stats
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    
    # Random erasing (cutout)
    transforms.RandomErasing(
        p=0.25,
        scale=(0.02, 0.15),
        ratio=(0.3, 3.3),
        value='random'
    )
])

# Validation/Test transforms (no augmentation)
val_transforms = transforms.Compose([
    transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print(f"✓ Training transforms defined for {config.IMAGE_SIZE}×{config.IMAGE_SIZE}")
print(f"✓ Validation transforms defined for {config.IMAGE_SIZE}×{config.IMAGE_SIZE}")
print("="*60)


In [ ]:
# Visualize augmentations
if len(train_df) > 0:
    sample_idx = np.random.randint(0, len(train_df))
    sample_path = train_df.iloc[sample_idx]['filepath']
    sample_age = train_df.iloc[sample_idx]['age']
    sample_label = train_df.iloc[sample_idx]['label_name']
    
    try:
        original_image = Image.open(sample_path).convert('RGB')
        
        print(f"Sample image: Age {sample_age} ({sample_label})")
        
        # Generate augmented versions
        fig, axes = plt.subplots(3, 5, figsize=(15, 9))
        axes = axes.flatten()
        
        # Original
        axes[0].imshow(original_image)
        axes[0].set_title(f'Original\nAge: {sample_age}', fontsize=10, fontweight='bold')
        axes[0].axis('off')
        
        # Augmented versions
        for i in range(1, 15):
            augmented_tensor = train_transforms(original_image)
            
            # Denormalize for visualization
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            denorm = augmented_tensor * std + mean
            denorm = torch.clamp(denorm, 0, 1)
            
            axes[i].imshow(denorm.permute(1, 2, 0))
            axes[i].set_title(f'Augmented {i}', fontsize=10)
            axes[i].axis('off')
        
        plt.suptitle(f'Training Augmentation Examples ({config.IMAGE_SIZE}×{config.IMAGE_SIZE})', 
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(config.OUTPUT_DIR, 'augmentation_examples.png'), 
                    dpi=200, bbox_inches='tight')
        plt.show()
        
        print("✓ Augmentation visualization saved")
    except Exception as e:
        print(f"⚠ Could not visualize augmentations: {e}")


## Step 10: PyTorch Dataset Class


In [ ]:
# ============================================================
# STEP 10: PYTORCH DATASET CLASS
# ============================================================

class AgeClassificationDataset(Dataset):
    """
    Custom Dataset for age-based binary classification
    """
    
    def __init__(self, dataframe, transform=None, return_metadata=False):
        """
        Args:
            dataframe: pandas DataFrame with columns ['filepath', 'label', 'age']
            transform: torchvision transforms
            return_metadata: if True, returns (image, label, metadata)
        """
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.return_metadata = return_metadata
        
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        # Get row
        row = self.dataframe.iloc[idx]
        
        img_path = row['filepath']
        label = row['label']
        age = row['age']
        
        # Load image
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            # Fallback for corrupted images
            if rank == 0:
                print(f"Warning: Error loading {img_path}: {e}")
            image = Image.new('RGB', (config.IMAGE_SIZE, config.IMAGE_SIZE), color=(128, 128, 128))
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        if self.return_metadata:
            metadata = {
                'filename': row['filename'],
                'age': age,
                'gender': row['gender'],
                'race': row['race']
            }
            return image, label, metadata
        else:
            return image, label

print("✓ AgeClassificationDataset class defined")


In [ ]:
# Create dataset instances
train_dataset = AgeClassificationDataset(
    train_df,
    transform=train_transforms,
    return_metadata=False
)

val_dataset = AgeClassificationDataset(
    val_df,
    transform=val_transforms,
    return_metadata=False
)

test_dataset = AgeClassificationDataset(
    test_df,
    transform=val_transforms,
    return_metadata=True  # Return metadata for analysis
)

print("="*60)
print("DATASET INSTANCES")
print("="*60)
print(f"Training dataset: {len(train_dataset):,} samples")
print(f"Validation dataset: {len(val_dataset):,} samples")
print(f"Test dataset: {len(test_dataset):,} samples")
print("="*60)

# Test loading
if rank == 0:
    print("\nTesting dataset loading...")
    sample_img, sample_label = train_dataset[0]
    print(f"✓ Training sample shape: {sample_img.shape}")
    print(f"✓ Label: {sample_label} ({config.CLASS_NAMES[sample_label]})")
    print(f"✓ Dtype: {sample_img.dtype}")
    print(f"✓ Value range: [{sample_img.min():.3f}, {sample_img.max():.3f}]")
    
    sample_img, sample_label = val_dataset[0]
    print(f"✓ Validation sample shape: {sample_img.shape}")
    
    sample_img, sample_label, metadata = test_dataset[0]
    print(f"✓ Test sample shape: {sample_img.shape}")
    print(f"✓ Metadata: {metadata}")
    print("\n✓ All datasets working correctly")


## Step 11: DataLoaders with DDP Support


In [ ]:
# ============================================================
# STEP 11: DATALOADERS WITH DDP SUPPORT
# ============================================================

print("="*60)
print("CREATING DATALOADERS")
print("="*60)

# Create DistributedSampler for training if using DDP (not needed for DataParallel)
if world_size > 1 and use_ddp:
    # DDP requires DistributedSampler to split data across processes
    train_sampler = DistributedSampler(
        train_dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
        seed=config.RANDOM_SEED
    )
    val_sampler = DistributedSampler(
        val_dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=False
    )
    shuffle_train = False  # Sampler handles shuffling
else:
    # DataParallel or single GPU - no sampler needed
    train_sampler = None
    val_sampler = None
    shuffle_train = True

# Training DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=shuffle_train,
    sampler=train_sampler,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY,
    drop_last=True  # Drop last incomplete batch for stable training
)

# Validation DataLoader
val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE * 2,  # Larger batch (no gradients)
    shuffle=False,
    sampler=val_sampler,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY,
    drop_last=False
)

# Test DataLoader (no sampler needed, only rank 0 evaluates)
test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE * 2,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY,
    drop_last=False
)

if rank == 0:
    print(f"✓ Training batches: {len(train_loader)}")
    print(f"✓ Validation batches: {len(val_loader)}")
    print(f"✓ Test batches: {len(test_loader)}")
    print(f"\nEstimated iterations per epoch: {len(train_loader)}")
    print(f"Total training iterations ({config.NUM_EPOCHS} epochs): {len(train_loader) * config.NUM_EPOCHS:,}")
    training_mode = 'DDP' if (world_size > 1 and use_ddp) else ('DataParallel' if world_size > 1 else 'Single GPU')
    print(f"Training mode: {training_mode}")
print("="*60)


In [ ]:
# Test DataLoader
if rank == 0:
    print("\nTesting DataLoader...")
    
    # Get a batch
    sample_batch = next(iter(train_loader))
    images, labels = sample_batch
    
    print(f"✓ Batch images shape: {images.shape}")
    print(f"✓ Batch labels shape: {labels.shape}")
    print(f"✓ Batch memory: {images.element_size() * images.nelement() / 1e6:.2f} MB")
    print(f"✓ Labels in batch: {labels.tolist()[:10]}")
    print(f"✓ Class distribution: under18={((labels==0).sum().item())}, adult={((labels==1).sum().item())}")
    
    print("\n✓ DataLoaders working correctly")


## Step 12: Model Definition with DDP


In [ ]:
# ============================================================
# STEP 12: MODEL DEFINITION WITH DDP
# ============================================================

print("="*60)
print("MODEL DEFINITION")
print("="*60)

# Load pre-trained EfficientNet-B3
if rank == 0:
    print(f"Loading {config.MODEL_NAME} (pre-trained: {config.PRETRAINED})...")

if config.PRETRAINED:
    weights = models.EfficientNet_B3_Weights.IMAGENET1K_V1
    model = models.efficientnet_b3(weights=weights)
else:
    model = models.efficientnet_b3(weights=None)

if rank == 0:
    print(f"✓ {config.MODEL_NAME} loaded")

# Get number of input features to classifier
num_features = model.classifier[1].in_features

if rank == 0:
    print(f"Original classifier input features: {num_features}")

# Replace classifier
# Note: inplace=False is required for mixed precision training (AMP) with DataParallel/DDP
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=False),
    nn.Linear(num_features, 512),
    nn.ReLU(inplace=False),  # inplace=False prevents gradient computation errors with AMP
    nn.Dropout(p=0.2, inplace=False),
    nn.Linear(512, config.NUM_CLASSES)
)

if rank == 0:
    print(f"✓ Classifier replaced: {num_features} → 512 → {config.NUM_CLASSES}")

# Move model to device
model = model.to(device)

# Use DDP or DataParallel for multi-GPU
if world_size > 1:
    if use_ddp:
        # Proper DDP (requires environment variables)
        model = DDP(model, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=False)
        if rank == 0:
            print(f"\n✓ Using DDP with {world_size} GPUs")
    else:
        # DataParallel (notebook-friendly, works without env vars)
        model = nn.DataParallel(model)
        if rank == 0:
            print(f"\n✓ Using DataParallel with {world_size} GPUs (notebook mode)")
else:
    if rank == 0:
        print(f"\n✓ Using single GPU: {device}")

# Count parameters
def count_parameters(model):
    """Count model parameters"""
    # Unwrap DDP or DataParallel
    if isinstance(model, (DDP, nn.DataParallel)):
        model = model.module
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

total_params, trainable_params = count_parameters(model)

if rank == 0:
    print(f"\nModel parameters:")
    print(f"  Total: {total_params:,}")
    print(f"  Trainable: {trainable_params:,}")
    print(f"  Non-trainable: {total_params - trainable_params:,}")
    print(f"  Model size: ~{total_params * 4 / 1e6:.1f} MB (FP32)")
    print("="*60)


## Step 13: Loss Function & Optimizer


In [ ]:
# ============================================================
# STEP 13: LOSS FUNCTION & OPTIMIZER
# ============================================================

print("="*60)
print("LOSS FUNCTION & OPTIMIZER")
print("="*60)

# Get model parameters (unwrap DDP or DataParallel if needed)
if isinstance(model, (DDP, nn.DataParallel)):
    model_params = model.module.parameters()
else:
    model_params = model.parameters()

# Class weights (prioritize under-18 recall)
class_weights = torch.tensor([1.5, 1.0]).to(device)  # [under18, adult]

# Loss function
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer
optimizer = optim.AdamW(
    model_params,
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY,
    betas=(0.9, 0.999)
)

# Learning rate scheduler
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='max',  # Maximize val recall
    factor=config.SCHEDULER_FACTOR,
    patience=config.SCHEDULER_PATIENCE,
    verbose=(rank == 0),  # Only print on rank 0
    min_lr=config.SCHEDULER_MIN_LR
)

# Mixed precision scaler
if config.USE_AMP:
    scaler = torch.cuda.amp.GradScaler()
    if rank == 0:
        print("✓ Mixed Precision (FP16) enabled")
else:
    scaler = None
    if rank == 0:
        print("✓ Using FP32 training")

if rank == 0:
    print(f"✓ Loss function: Weighted CrossEntropyLoss")
    print(f"  Class weights: under18={class_weights[0].item()}, adult={class_weights[1].item()}")
    print(f"✓ Optimizer: AdamW")
    print(f"  Base Learning Rate: {config.BASE_LEARNING_RATE}")
    print(f"  Scaled Learning Rate: {config.LEARNING_RATE} (×{world_size} for {world_size} GPU{'s' if world_size > 1 else ''})")
    print(f"  Effective Batch Size: {config.BATCH_SIZE * world_size}")
    print(f"  LR per Sample: {config.LEARNING_RATE / (config.BATCH_SIZE * world_size):.2e}")
    print(f"  Weight decay: {config.WEIGHT_DECAY}")
    print(f"✓ Scheduler: ReduceLROnPlateau")
    print(f"  Mode: maximize val_under18_recall")
    print(f"  Patience: {config.SCHEDULER_PATIENCE} epochs")
    print(f"  Factor: {config.SCHEDULER_FACTOR}")
    print(f"  Min LR: {config.SCHEDULER_MIN_LR}")
    if world_size > 1:
        print(f"\n  📊 LR Scaling Rationale:")
        print(f"     - With {world_size} GPUs, effective batch size = {config.BATCH_SIZE * world_size}")
        print(f"     - Linear scaling: LR = base_lr × num_gpus = {config.BASE_LEARNING_RATE} × {world_size} = {config.LEARNING_RATE}")
        print(f"     - This maintains same learning rate per sample across different batch sizes")
print("="*60)


In [ ]:
# ============================================================
# STEP 14: TRAINING UTILITIES
# ============================================================

def calculate_metrics(y_true, y_pred):
    """Calculate classification metrics"""
    accuracy = accuracy_score(y_true, y_pred)
    
    # Per-class metrics
    precision_per_class = precision_score(y_true, y_pred, average=None, zero_division=0)
    recall_per_class = recall_score(y_true, y_pred, average=None, zero_division=0)
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    
    # Overall metrics
    precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'under18_precision': precision_per_class[0],
        'under18_recall': recall_per_class[0],
        'under18_f1': f1_per_class[0],
        'adult_precision': precision_per_class[1],
        'adult_recall': recall_per_class[1],
        'adult_f1': f1_per_class[1],
        'macro_precision': precision_macro,
        'macro_recall': recall_macro,
        'macro_f1': f1_macro
    }

def print_metrics(metrics, prefix=""):
    """Pretty print metrics"""
    print(f"\n{prefix}Metrics:")
    print(f"  Accuracy: {metrics['accuracy']*100:.2f}%")
    print(f"  Under-18: Precision={metrics['under18_precision']*100:.2f}%, "
          f"Recall={metrics['under18_recall']*100:.2f}%, F1={metrics['under18_f1']:.4f}")
    print(f"  Adult: Precision={metrics['adult_precision']*100:.2f}%, "
          f"Recall={metrics['adult_recall']*100:.2f}%, F1={metrics['adult_f1']:.4f}")

if rank == 0:
    print("✓ Utility functions defined")


## Step 15: Training Loop with DDP


In [ ]:
# ============================================================
# STEP 15: TRAINING LOOP WITH DDP
# ============================================================

# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_under18_recall': [],
    'val_adult_recall': [],
    'learning_rate': []
}

# Best model tracking
best_val_recall = 0.0
best_epoch = 0
patience_counter = 0

# Checkpoint directory
checkpoint_dir = os.path.join(config.OUTPUT_DIR, 'checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)

if rank == 0:
    print("="*60)
    print("STARTING TRAINING")
    print("="*60)
    print(f"Total epochs: {config.NUM_EPOCHS}")
    print(f"Early stopping patience: {config.EARLY_STOPPING_PATIENCE}")
    print(f"Monitoring metric: {config.METRIC_TO_MONITOR}")
    print("="*60)

# Training loop
for epoch in range(config.NUM_EPOCHS):
    epoch_start_time = time.time()
    
    # Set epoch for DistributedSampler (only needed for DDP)
    if world_size > 1 and use_ddp and train_sampler is not None:
        train_sampler.set_epoch(epoch)
    
    # ========== TRAINING PHASE ==========
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    all_train_preds = []
    all_train_labels = []
    
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config.NUM_EPOCHS} [Train]', 
                      disable=(rank != 0))
    
    for batch_idx, (images, labels) in enumerate(train_pbar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass with mixed precision
        if config.USE_AMP:
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            # Backward pass
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        # Statistics
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
        
        # Collect predictions for metrics
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_labels.extend(labels.cpu().numpy())
        
        # Update progress bar
        if rank == 0:
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100*train_correct/train_total:.2f}%'
            })
    
    # Average training loss
    train_loss /= len(train_loader)
    train_acc = train_correct / train_total
    
    # ========== VALIDATION PHASE ==========
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    all_val_preds = []
    all_val_labels = []
    
    val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{config.NUM_EPOCHS} [Val]', 
                    disable=(rank != 0))
    
    with torch.no_grad():
        for images, labels in val_pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            # Forward pass
            if config.USE_AMP:
                with torch.cuda.amp.autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            # Statistics
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
            # Collect predictions
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())
            
            if rank == 0:
                val_pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{100*val_correct/val_total:.2f}%'
                })
    
    # Average validation loss
    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    
    # Calculate detailed metrics (only on rank 0)
    if rank == 0:
        val_metrics = calculate_metrics(all_val_labels, all_val_preds)
        val_under18_recall = val_metrics['under18_recall']
        
        # Update history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_under18_recall'].append(val_under18_recall)
        history['val_adult_recall'].append(val_metrics['adult_recall'])
        history['learning_rate'].append(optimizer.param_groups[0]['lr'])
        
        # Learning rate scheduling
        scheduler.step(val_under18_recall)
        
        # Checkpoint saving
        is_best = val_under18_recall > best_val_recall
        if is_best:
            best_val_recall = val_under18_recall
            best_epoch = epoch + 1
            patience_counter = 0
            
            # Save best model (unwrap DDP or DataParallel if needed)
            if isinstance(model, (DDP, nn.DataParallel)):
                model_state = model.module.state_dict()
            else:
                model_state = model.state_dict()
            
            checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model_state,
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_val_recall': best_val_recall,
                'val_metrics': val_metrics,
                'config': config.__dict__
            }
            
            torch.save(checkpoint, os.path.join(checkpoint_dir, 'best_model.pth'))
            print(f"\n✓ Best model saved! Val Under-18 Recall: {best_val_recall*100:.2f}%")
        else:
            patience_counter += 1
        
        # Epoch summary
        epoch_time = time.time() - epoch_start_time
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}/{config.NUM_EPOCHS} Summary")
        print(f"{'='*60}")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")
        print_metrics(val_metrics, prefix="Val ")
        print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
        print(f"Time: {epoch_time:.2f}s")
        print(f"Best Epoch: {best_epoch} (Recall: {best_val_recall*100:.2f}%)")
        print(f"Early Stopping: {patience_counter}/{config.EARLY_STOPPING_PATIENCE}")
        print(f"{'='*60}\n")
        
        # Early stopping
        if patience_counter >= config.EARLY_STOPPING_PATIENCE:
            print(f"\n⚠ Early stopping triggered! No improvement for {config.EARLY_STOPPING_PATIENCE} epochs.")
            print(f"Best model was at epoch {best_epoch} with Val Under-18 Recall: {best_val_recall*100:.2f}%")
            break
    
    # Synchronize all processes (only needed for DDP)
    if world_size > 1 and use_ddp:
        dist.barrier()

if rank == 0:
    print("\n" + "="*60)
    print("TRAINING COMPLETED")
    print("="*60)
    print(f"Best epoch: {best_epoch}")
    print(f"Best Val Under-18 Recall: {best_val_recall*100:.2f}%")
    print("="*60)


In [ ]:
# ============================================================
# STEP 16: TRAINING HISTORY VISUALIZATION
# ============================================================

if rank == 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch', fontsize=12)
    axes[0, 0].set_ylabel('Loss', fontsize=12)
    axes[0, 0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    
    # Accuracy
    axes[0, 1].plot(epochs, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
    axes[0, 1].plot(epochs, history['val_acc'], 'r-', label='Val Acc', linewidth=2)
    axes[0, 1].set_xlabel('Epoch', fontsize=12)
    axes[0, 1].set_ylabel('Accuracy', fontsize=12)
    axes[0, 1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)
    
    # Under-18 Recall
    axes[0, 2].plot(epochs, history['val_under18_recall'], 'g-', label='Val Under-18 Recall', linewidth=2)
    axes[0, 2].axhline(y=best_val_recall, color='r', linestyle='--', label=f'Best: {best_val_recall*100:.2f}%')
    axes[0, 2].set_xlabel('Epoch', fontsize=12)
    axes[0, 2].set_ylabel('Recall', fontsize=12)
    axes[0, 2].set_title('Validation Under-18 Recall', fontsize=14, fontweight='bold')
    axes[0, 2].legend()
    axes[0, 2].grid(alpha=0.3)
    
    # Adult Recall
    axes[1, 0].plot(epochs, history['val_adult_recall'], 'm-', label='Val Adult Recall', linewidth=2)
    axes[1, 0].set_xlabel('Epoch', fontsize=12)
    axes[1, 0].set_ylabel('Recall', fontsize=12)
    axes[1, 0].set_title('Validation Adult Recall', fontsize=14, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)
    
    # Learning Rate
    axes[1, 1].plot(epochs, history['learning_rate'], 'orange', linewidth=2)
    axes[1, 1].set_xlabel('Epoch', fontsize=12)
    axes[1, 1].set_ylabel('Learning Rate', fontsize=12)
    axes[1, 1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(alpha=0.3)
    
    # Combined metrics
    axes[1, 2].plot(epochs, history['val_under18_recall'], 'g-', label='Under-18 Recall', linewidth=2)
    axes[1, 2].plot(epochs, history['val_adult_recall'], 'm-', label='Adult Recall', linewidth=2)
    axes[1, 2].plot(epochs, history['val_acc'], 'b-', label='Accuracy', linewidth=2)
    axes[1, 2].set_xlabel('Epoch', fontsize=12)
    axes[1, 2].set_ylabel('Score', fontsize=12)
    axes[1, 2].set_title('All Validation Metrics', fontsize=14, fontweight='bold')
    axes[1, 2].legend()
    axes[1, 2].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(config.OUTPUT_DIR, 'training_history.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Training history visualization saved")


## Step 17: Load Best Model & Test Evaluation


In [ ]:
# ============================================================
# STEP 17: LOAD BEST MODEL & TEST EVALUATION
# ============================================================

if rank == 0:
    print("="*60)
    print("LOADING BEST MODEL")
    print("="*60)
    
    # Load best checkpoint
    checkpoint_path = os.path.join(checkpoint_dir, 'best_model.pth')
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Load model state (unwrap DDP or DataParallel if needed)
    if isinstance(model, (DDP, nn.DataParallel)):
        model.module.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint['model_state_dict'])
    
    print(f"✓ Best model loaded from epoch {checkpoint['epoch']}")
    print(f"✓ Best Val Under-18 Recall: {checkpoint['best_val_recall']*100:.2f}%")
    
    # Test evaluation
    print("\n" + "="*60)
    print("TEST SET EVALUATION")
    print("="*60)
    
    model.eval()
    all_test_preds = []
    all_test_labels = []
    all_test_metadata = []
    
    test_pbar = tqdm(test_loader, desc='Testing')
    
    with torch.no_grad():
        for images, labels, metadata in test_pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            if config.USE_AMP:
                with torch.cuda.amp.autocast():
                    outputs = model(images)
            else:
                outputs = model(images)
            
            _, predicted = torch.max(outputs.data, 1)
            
            all_test_preds.extend(predicted.cpu().numpy())
            all_test_labels.extend(labels.cpu().numpy())
            all_test_metadata.extend(metadata)
    
    # Calculate test metrics
    test_metrics = calculate_metrics(all_test_labels, all_test_preds)
    
    print("\n" + "="*60)
    print("TEST SET RESULTS")
    print("="*60)
    print_metrics(test_metrics, prefix="Test ")
    print("="*60)
    
    # Confusion Matrix
    cm = confusion_matrix(all_test_labels, all_test_preds)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=config.CLASS_NAMES, 
                yticklabels=config.CLASS_NAMES)
    plt.xlabel('Predicted', fontsize=12)
    plt.ylabel('Actual', fontsize=12)
    plt.title('Test Set Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(config.OUTPUT_DIR, 'test_confusion_matrix.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Confusion matrix saved")
    
    # Classification Report
    print("\n" + "="*60)
    print("DETAILED CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(all_test_labels, all_test_preds, 
                                target_names=config.CLASS_NAMES, digits=4))
    print("="*60)


## Step 18: Cleanup DDP (if used)


In [ ]:
# ============================================================
# STEP 18: CLEANUP DDP
# ============================================================

# Cleanup DDP process group (only if DDP was used)
if world_size > 1 and use_ddp:
    dist.destroy_process_group()
    if rank == 0:
        print("✓ DDP process group destroyed")

if rank == 0:
    print("\n" + "="*60)
    print("PIPELINE COMPLETE!")
    print("="*60)
    print(f"✓ Best model saved: {checkpoint_dir}/best_model.pth")
    print(f"✓ Training history: training_history.png")
    print(f"✓ Test results: test_confusion_matrix.png")
    print("="*60)
